# Pipe Endpoint Detection by Voxel Graph

This notebook visualizes pipe end candidates detected from a coarse voxel graph. It is separate from the cylinder segmentation notebook.

In [9]:
from pathlib import Path
import sys
import json
import importlib

import numpy as np
import open3d as o3d

PLUGIN_ROOT = Path.cwd()
if PLUGIN_ROOT.name != "poseDeterminator":
    PLUGIN_ROOT = Path("python/plugins/poseDeterminator").resolve()

if str(PLUGIN_ROOT) not in sys.path:
    sys.path.insert(0, str(PLUGIN_ROOT))

import PipeEndProfileAnalyzer as pipe_analyzer_module
pipe_analyzer_module = importlib.reload(pipe_analyzer_module)
analyze_pipe_endpoints_by_voxel_graph = pipe_analyzer_module.analyze_pipe_endpoints_by_voxel_graph

PLUGIN_ROOT


WindowsPath('d:/flame_robotics_drt/python/plugins/poseDeterminator')

## Settings

`GRAPH_VOXEL_SIZE` is the main parameter. Use `None` to auto-pick a value from the pipe bounding box. If the graph is disconnected, increase it. If the endpoint is too coarse, decrease it.

In [10]:
PIPE_PATH = PLUGIN_ROOT / "data" / "PIPE NO.1.ply"
# PIPE_PATH = PLUGIN_ROOT / "data" / "PIPE NO.3_fill.ply"

SCALE = 1.0
MAX_POINTS = 50000

GRAPH_VOXEL_SIZE = None  # auto; for mm-scale samples try 40~100, for m-scale samples try 0.04~0.10
GRAPH_MIN_VOXEL_POINTS = 2
GRAPH_NEIGHBOR_RADIUS = 1
MAX_ENDPOINT_CANDIDATES = 8
MIN_BRANCH_LENGTH = None  # auto; increase to suppress short/noisy branch tips
MOUNT_BACKOFF_DISTANCE = None  # auto; distance to step inward from a graph endpoint
MOUNT_STRAIGHT_WINDOW = None  # auto; local path length used for straightness check
MOUNT_MIN_STRAIGHTNESS = 0.92

VISUAL_MAX_POINTS = 20000

assert PIPE_PATH.exists(), PIPE_PATH
PIPE_PATH


WindowsPath('d:/flame_robotics_drt/python/plugins/poseDeterminator/data/PIPE NO.1.ply')

## Run Detector

The detector builds a voxel graph, keeps the largest connected component, and selects the graph-diameter endpoints.

In [11]:
pipe_analyzer_module = importlib.reload(pipe_analyzer_module)
analyze_pipe_endpoints_by_voxel_graph = pipe_analyzer_module.analyze_pipe_endpoints_by_voxel_graph

graph_result = analyze_pipe_endpoints_by_voxel_graph(
    PIPE_PATH,
    scale=SCALE,
    voxel_size=GRAPH_VOXEL_SIZE,
    max_points=MAX_POINTS,
    min_voxel_points=GRAPH_MIN_VOXEL_POINTS,
    neighbor_radius_voxels=GRAPH_NEIGHBOR_RADIUS,
    max_endpoint_candidates=MAX_ENDPOINT_CANDIDATES,
    min_branch_length=MIN_BRANCH_LENGTH,
    mount_backoff_distance=MOUNT_BACKOFF_DISTANCE,
    mount_straight_window=MOUNT_STRAIGHT_WINDOW,
    mount_min_straightness=MOUNT_MIN_STRAIGHTNESS,
    include_points=False,
    random_seed=0,
)

summary = {
    "voxel_size": graph_result["voxel_size"],
    "point_count": graph_result["point_count"],
    "fit_point_count": graph_result["fit_point_count"],
    "node_count": graph_result["node_count"],
    "component_node_count": graph_result["component_node_count"],
    "edge_count": graph_result["edge_count"],
    "diameter_distance": graph_result["diameter_distance"],
    "terminal_node_indices": graph_result["terminal_node_indices"],
    "terminal_end_points": graph_result["terminal_end_points"],
    "min_branch_length": graph_result["min_branch_length"],
    "mount_backoff_distance": graph_result["mount_backoff_distance"],
    "mount_straight_window": graph_result["mount_straight_window"],
    "mount_min_straightness": graph_result["mount_min_straightness"],
    "endpoint_candidates": graph_result["endpoint_candidates"],
    "mount_candidates": graph_result["mount_candidates"],
    "timing_sec": graph_result["timing_sec"],
}
print(json.dumps(summary, indent=2, ensure_ascii=False))


{
  "voxel_size": 45.475741563466556,
  "point_count": 500000,
  "fit_point_count": 50000,
  "node_count": 239,
  "component_node_count": 239,
  "edge_count": 1399,
  "diameter_distance": 1942.2269947298314,
  "terminal_node_indices": [
    238,
    53
  ],
  "terminal_end_points": [
    [
      604.1498781558457,
      -1513.688015505244,
      26.15949528548151
    ],
    [
      -21.89403168114665,
      -17.728735721030517,
      24.9519482926155
    ]
  ],
  "min_branch_length": 68.21361234519983,
  "mount_backoff_distance": 90.95148312693311,
  "mount_straight_window": 136.42722469039967,
  "mount_min_straightness": 0.92,
  "endpoint_candidates": [
    {
      "node_index": 238,
      "point": [
        604.1498781558457,
        -1513.688015505244,
        26.15949528548151
      ],
      "score": 1.0,
      "branch_distance": 0.0,
      "reasons": [
        "diameter_endpoint"
      ],
      "rank": 0
    },
    {
      "node_index": 53,
      "point": [
        -21.89403168114

## Visualize

Gray: original pipe points. Blue: voxel graph. Yellow: graph-diameter path. Red/green: diameter endpoints. Purple: endpoint candidates. Cyan: chuck mount candidates selected from local straightness.

In [12]:
def make_lines(points, lines, color):
    line_set = o3d.geometry.LineSet()
    line_set.points = o3d.utility.Vector3dVector(np.asarray(points, dtype=float))
    line_set.lines = o3d.utility.Vector2iVector(np.asarray(lines, dtype=np.int32))
    line_set.paint_uniform_color(color)
    return line_set


context_pcd = o3d.io.read_point_cloud(str(PIPE_PATH))
if SCALE != 1.0:
    context_pcd.scale(SCALE, center=np.zeros(3))
if len(context_pcd.points) > VISUAL_MAX_POINTS:
    context_pcd = context_pcd.random_down_sample(VISUAL_MAX_POINTS / len(context_pcd.points))
context_pcd.paint_uniform_color([0.72, 0.72, 0.72])

nodes = np.asarray(graph_result["graph"]["nodes"], dtype=float)
edges = graph_result["graph"]["edges"]
path_node_indices = graph_result["graph"]["path_node_indices"]
terminal_points = np.asarray(graph_result["terminal_end_points"], dtype=float)
endpoint_candidates = graph_result.get("endpoint_candidates", [])
mount_candidates = graph_result.get("mount_candidates", [])

node_pcd = o3d.geometry.PointCloud()
node_pcd.points = o3d.utility.Vector3dVector(nodes)
node_pcd.paint_uniform_color([0.0, 0.2, 1.0])

bbox_extent = context_pcd.get_axis_aligned_bounding_box().get_extent()
effective_voxel_size = float(graph_result["voxel_size"])
marker_radius = max(float(np.linalg.norm(bbox_extent)) * 0.012, effective_voxel_size * 0.35)

geometries = [context_pcd, node_pcd]
if edges:
    geometries.append(make_lines(nodes, edges, [0.0, 0.45, 1.0]))

if len(path_node_indices) >= 2:
    path_points = nodes[path_node_indices]
    path_lines = [[i, i + 1] for i in range(len(path_points) - 1)]
    geometries.append(make_lines(path_points, path_lines, [1.0, 0.85, 0.0]))

endpoint_colors = [[1.0, 0.0, 0.0], [0.0, 0.8, 0.2]]
for idx, point in enumerate(terminal_points):
    sphere = o3d.geometry.TriangleMesh.create_sphere(radius=marker_radius, resolution=24)
    sphere.translate(point)
    sphere.paint_uniform_color(endpoint_colors[idx % len(endpoint_colors)])
    geometries.append(sphere)

for candidate in endpoint_candidates:
    if "diameter_endpoint" in candidate.get("reasons", []):
        continue
    point = np.asarray(candidate["point"], dtype=float)
    sphere = o3d.geometry.TriangleMesh.create_sphere(radius=marker_radius * 0.75, resolution=20)
    sphere.translate(point)
    sphere.paint_uniform_color([0.65, 0.0, 1.0])
    geometries.append(sphere)

for candidate in mount_candidates:
    mount_point = np.asarray(candidate["mount_point"], dtype=float)
    endpoint_point = np.asarray(candidate["endpoint_point"], dtype=float)
    sphere = o3d.geometry.TriangleMesh.create_sphere(radius=marker_radius * 0.85, resolution=20)
    sphere.translate(mount_point)
    sphere.paint_uniform_color([0.0, 0.85, 1.0])
    geometries.append(sphere)
    geometries.append(make_lines([endpoint_point, mount_point], [[0, 1]], [0.0, 0.85, 1.0]))

o3d.visualization.draw_geometries(geometries)
